In [1]:
puts `ls -l rawdata`

total 7716
-rw-rw-r-- 1 osboxes osboxes 3678593 Aug 12 17:45 disease-gene.csv
-rw-rw-r-- 1 osboxes osboxes 1419434 Aug 12 17:45 disease-therapeuticarea.csv
-rw-rw-r-- 1 osboxes osboxes   50286 Aug 12 17:45 drug-disease.csv
-rw-rw-r-- 1 osboxes osboxes   97134 Aug 12 17:45 drug-drugtype.csv
-rw-rw-r-- 1 osboxes osboxes  148014 Aug 12 17:45 drug-gene.csv
-rw-rw-r-- 1 osboxes osboxes     576 Aug 12 17:45 edges.csv
-rw-rw-r-- 1 osboxes osboxes  201313 Aug 12 17:45 gene-reactome.csv
-rw-rw-r-- 1 osboxes osboxes 2282825 Aug 12 17:45 Graph_edge_info.xlsx


In [1]:
puts `head -2 ./rawdata/drug-disease.csv`

"source","source_type","target","target_type"
"CHEMBL1200656","drug","EFO_1001888","disease"


In [2]:
puts `ls -lt maps`

total 1040
-rw-rw-r-- 1 osboxes osboxes  95623 Jun 13 14:23 diseases-errors.txt
-rw-rw-r-- 1 osboxes osboxes 536532 Jun 13 14:23 diseases.map
-rw-rw-r-- 1 osboxes osboxes 135160 Jun 10 17:05 genes.map.bak
-rw-rw-r-- 1 osboxes osboxes 150762 Jun 10 17:05 genes.map
-rw-rw-r-- 1 osboxes osboxes  16053 Feb 17 15:05 no_mondo_mapping.txt
-rw-rw-r-- 1 osboxes osboxes  92125 Jan 22 10:05 drugs.map
-rw-rw-r-- 1 osboxes osboxes   2029 Jan 22 10:05 drugs-biologics-errors.txt
-rw-rw-r-- 1 osboxes osboxes  11009 Jan  7 13:20 drugs-errors.txt
drwxrwxr-x 2 osboxes osboxes   4096 Jan  7 13:20 backup_pre_NYE
drwxrwxr-x 2 osboxes osboxes   4096 Jan  7 13:20 DEPRECATED


In [3]:
puts `head -2 ./maps/drugs.map`

chembl,label,CID,IUPACname
CHEMBL1200656,CHEMBL1200656,5284447,Natamycin


### note that the CID column will be prefixed with "SUBSTANCE_" for biologics, and the URL should be /substances/
```
CHEMBL1201632,CHEMBL1201632,SUBSTANCE_50069655,Cerezyme
CHEMBL2108078,CHEMBL2108078,SUBSTANCE_135651867,Natpar
```

In [4]:
puts `head -2 ./maps/diseases.map`

source,mondo,prefname
MONDO_0000030,http://purl.obolibrary.org/obo/MONDO_0000030,familial sleep-related hypermotor epilepsy


In [6]:
puts `grep Orphanet ./maps/diseases.map | head -5`

Orphanet_10,http://purl.obolibrary.org/obo/MONDO_0015028,"48,XXYY syndrome"
Orphanet_100,http://purl.obolibrary.org/obo/MONDO_0008840,ataxia telangiectasia
Orphanet_1000,http://purl.obolibrary.org/obo/MONDO_0010390,ocular albinism with late-onset sensorineural deafness
Orphanet_100006,http://purl.obolibrary.org/obo/MONDO_0015033,"ABeta amyloidosis, dutch type"
Orphanet_100008,http://purl.obolibrary.org/obo/MONDO_0007098,ACys amyloidosis


In [7]:
require 'linkeddata'
require 'rdf/nquads'
require 'csv'

e = File.open('./graph/2026_drug-disease-errors.txt', 'w') 

# Define namespaces
SIMPATHIC = RDF::Vocabulary.new('urn:simpathic:')
RDFS = RDF::Vocabulary.new('http://www.w3.org/2000/01/rdf-schema#')

# Read input files
drug_mappings = CSV.read('./maps/drugs.map', headers: true)
disease_mappings = CSV.read('./maps/diseases.map', headers: true)  # Orphanet has a ":", other have a "_"
failures = {}

# refresh
f = File.open('./graph/2026_drug-disease.nq.large', 'w')
f.close

recordcount = 0
CSV.foreach('./rawdata/drug-disease.csv', col_sep: ",", quote_char: '"', 
  liberal_parsing: true, headers: true) do |row|
  # "source","source_type","target","target_type"
  # "CHEMBL1200656","drug","EFO_1001888","disease"
  drug_id = row['source']
  disease_id = row['target']

#   warn "searching for #{drug_id}"
  # chembl,label,CID,IUPACname
  # CHEMBL1200656,CHEMBL1200656,5284447,Natamycin
  # CHEMBL1201632,CHEMBL1201632,SUBSTANCE_50069655,Cerezyme
  drug = drug_mappings.find { |d| d['chembl'] == drug_id }

# source,mondo,prefname
# Orphanet:100032,MONDO:0968955,Hypocalcified amelogenesis imperfecta
  disease = disease_mappings.find { |d| d['source'] == disease_id }
  
#     warn "found #{drug}"
  unless drug
    next if failures[drug_id]
    failures[drug_id] = 1
    warn "drug lookup failed #{drug_id}"
    e.write "drug lookup failed #{drug_id}\n"
    next
  end
  unless disease
    next if failures[disease_id]
    failures[disease_id] = 1
    warn "disease lookup failed #{disease_id}"
    e.write "disease lookup failed #{disease_id}\n"
    next
  end
  
  
  # Extract relevant IDs and labels
  # chembl,label,CID,IUPACname
  # CHEMBL1200656,CHEMBL1200656,5284447,Natamycin
  # CHEMBL1201632,CHEMBL1201632,SUBSTANCE_50069655,Cerezyme
  drugid = drug['CID']
  if drugid =~ /SUBSTANCE_(\d+)/
      pubchem_uri = RDF::URI.new("https://pubchem.ncbi.nlm.nih.gov/substance/#{$1}")
  else
      pubchem_uri = RDF::URI.new("https://pubchem.ncbi.nlm.nih.gov/compound/#{drug['CID']}")
  end
  pubchem_type = RDF::URI.new("http://semanticscience.org/resource/CHEMINF_000302")
  pubchem_label =  RDF::Literal.new("PubChem Identifier")
  pubchem_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Drug")
  drug_label = RDF::Literal.new(drug['label'])
  iupac_drug_label = RDF::Literal.new(drug['IUPACname'])
  original_drug = RDF::Literal.new(drug['chembl'])



# source,mondo,prefname
# EFO_0000174,http://purl.obolibrary.org/obo/MONDO_0012817,Ewing sarcoma
  mondo_uri = RDF::URI.new(disease['mondo'])  
  mondo_type = RDF::URI.new("https://bioportal.bioontology.org/ontologies/MONDO")
  mondo_core_type = RDF::URI.new("https://w3id.org/biolink/vocab/Disease")
  mondo_label =  RDF::Literal.new("MONDO Term")
#   orphanet = RDF::URI.new(disease['orpha'])
  disease_label = RDF::Literal.new(disease['prefname'])
  original_disease = RDF::Literal.new(disease['source'])
  
  # Create context URI
  context_uri = RDF::URI.new("urn:simpathic:context:#{drug_id}_#{disease_id}")
  general_context = RDF::URI.new("urn:simpathic:context:all_metadata")
  
  # Create RDF repository (need to do this each time, since there are hundreds of thousands of lines, and the graph gets too big for memory)
  graph = RDF::Repository.new

  # Add quads to graph using RDF::Statement
  graph << RDF::Statement.new(pubchem_uri, SIMPATHIC['associated-with'], mondo_uri, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_uri, SIMPATHIC['associated-with'], pubchem_uri, graph_name: context_uri)
  
  # graph << RDF::Statement.new(pubchem_uri,  RDFS.label,     drug_label, graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_uri,  RDFS.label,     iupac_drug_label, graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_uri,  RDF.type,       pubchem_type,          graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_uri,  RDF.type,       pubchem_core_type,             graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_type, RDFS.label,     RDF::Literal.new("PubChem"), graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_core_type, RDFS.label,     RDF::Literal.new("Drug"), graph_name: context_uri)
  graph << RDF::Statement.new(pubchem_uri,  SIMPATHIC['original-id'], original_drug, graph_name: context_uri)
 
  graph << RDF::Statement.new(mondo_uri, RDFS.label, disease_label, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_uri, RDF.type, mondo_type, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_uri, RDF.type, mondo_core_type, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_type, RDFS.label, mondo_label, graph_name: context_uri)
#   graph << RDF::Statement.new(mondo_uri, SIMPATHIC['orphanet'], orphanet, graph_name: context_uri)
  graph << RDF::Statement.new(mondo_uri, SIMPATHIC['original-id'], original_disease, graph_name: context_uri)
  
  graph << RDF::Statement.new(context_uri, SIMPATHIC['skg-source'], RDF::Literal.new("Radboud"), graph_name: general_context)
  #   graph << RDF::Statement.new(context_uri, SIMPATHIC['skg-evidence'], RDF::Literal.new("TBD"), graph_name: context_uri)
  
  # Write RDF to file in N-Quads format
  File.open('./graph/2026_drug-disease.nq.large', 'a') do |f|
    RDF::Writer.for(:nquads).new(f) do |writer|
      writer << graph
    end
  end
end

e.close

puts "RDF quads written"

disease lookup failed EFO_1001888
disease lookup failed EFO_0000211
disease lookup failed HP_0000103
disease lookup failed EFO_1000781
disease lookup failed EFO_0004220
disease lookup failed EFO_0009552
disease lookup failed EFO_0005752
disease lookup failed EFO_0003894
disease lookup failed HP_0012735
disease lookup failed HP_0001742
disease lookup failed HP_0100607
disease lookup failed EFO_0010072
disease lookup failed EFO_0007214
disease lookup failed EFO_0000180
disease lookup failed EFO_0004239
disease lookup failed HP_0004419
disease lookup failed EFO_0003907
disease lookup failed EFO_0003956
disease lookup failed EFO_0004274
disease lookup failed EFO_1001896
drug lookup failed CHEMBL2108222
disease lookup failed EFO_0003929
disease lookup failed EFO_1001069
disease lookup failed EFO_1001485
disease lookup failed HP_0002069
disease lookup failed EFO_1000760
disease lookup failed EFO_0005687
disease lookup failed EFO_0007149
disease lookup failed EFO_1001459
disease lookup failed

RDF quads written
